In [1]:
import os
import sys
sys.path.append("/home/amirabbas-kazeminia/Projects/DeepRT")
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import torch

from Model.deeprt import DeepRT
model = DeepRT.load_from_checkpoint("/home/amirabbas-kazeminia/Projects/DeepRT/weights/2layers_best_model_10.ckpt")

In [2]:
import pandas as pd
data = pd.read_csv("../data/ML_DATA.csv")
data_ = data.T.to_dict()
mapping = {datapoint["Peptide"]:datapoint['B'] for datapoint in data_.values()}
sequences = data["Peptide"].to_list()

In [3]:
pairs = []
threshold = 2
for seq in sequences:
    if seq[0]=="f" and seq[-2:]=="Kf":
        pair0 = seq
        pair1 = "F"+seq[1:-1]+"F"
        if pair1 in sequences:
            if abs(mapping[pair0]-mapping[pair1])>threshold:
                pairs.append((pair0, pair1))
    elif seq[0]=='f' and seq[-1]=='R':
        pair0 = seq
        pair1 = "F"+seq[1:]
        if pair1 in sequences:
            if abs(mapping[pair0]-mapping[pair1])>threshold:
                pairs.append((pair0, pair1))
real_deltas = []
for pair in pairs:
    real_deltas.append(mapping[pair[1]]-mapping[pair[0]])

In [4]:
f_seqs = [pair[0] for pair in pairs]
F_seqs = [pair[1] for pair in pairs]

In [5]:
f_scores = model.predict(f_seqs)["B_scores"]
F_scores = model.predict(F_seqs)["B_scores"]

In [6]:
pred = F_scores - f_scores

In [7]:
acc = (np.array((pred>0))==(np.array(real_deltas)>0)).sum()

In [8]:
acc/len(real_deltas)

np.float64(0.7015190372854606)